In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# =========================
# CONFIG
# =========================
QUICK_RUN = False

if QUICK_RUN:
    EPOCHS_A = 8
    EPOCHS_B = 12
    BATCH_SIZE = 128
    MEMORY_SIZE = 2000
    NUM_WORKERS = 2
    SEEDS = [0, 1, 2]
else:
    EPOCHS_A = 50
    EPOCHS_B = 100
    BATCH_SIZE = 128
    MEMORY_SIZE = 2000
    NUM_WORKERS = 2
    SEEDS = [0, 1, 2, 3, 4]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# REPRODUCIBILITY
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# =========================
# DATA
# =========================
stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))

train_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

def class_subset(dataset, class_range):
    idx = [i for i, y in enumerate(dataset.targets) if y in class_range]
    return Subset(dataset, idx)

# =========================
# MODEL (plain ResNet18)
# =========================
class CIFARResNet18(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        m = torchvision.models.resnet18(weights=None, num_classes=num_classes)
        m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        self.net = m

    def forward(self, x):
        return self.net(x)

# =========================
# VANILLA ER BUFFER (Reservoir)
# =========================
class ReservoirBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.n_seen = 0
        self.images = []
        self.labels = []

    def add_batch(self, x, y):
        x_cpu = x.detach().cpu()
        y_cpu = y.detach().cpu()
        for i in range(x_cpu.size(0)):
            self.n_seen += 1
            if len(self.images) < self.capacity:
                self.images.append(x_cpu[i].clone())
                self.labels.append(y_cpu[i].clone())
            else:
                j = random.randint(0, self.n_seen - 1)
                if j < self.capacity:
                    self.images[j] = x_cpu[i].clone()
                    self.labels[j] = y_cpu[i].clone()

    def can_sample(self, batch_size):
        return len(self.images) >= batch_size

    def sample(self, batch_size, device):
        idx = np.random.choice(len(self.images), size=batch_size, replace=False)
        x = torch.stack([self.images[i] for i in idx]).to(device)
        y = torch.stack([self.labels[i] for i in idx]).to(device)
        return x, y

# =========================
# EVAL
# =========================
def evaluate(model, loader, mode="all"):
    """
    mode:
      - all: true CIL (argmax over 100 classes)
      - A  : task-aware A eval (argmax over 0..49)
      - B  : task-aware B eval (argmax over 50..99)
    """
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)

            if mode == "A":
                pred = logits[:, :50].argmax(1)
            elif mode == "B":
                pred = logits[:, 50:].argmax(1) + 50
            else:
                pred = logits.argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

# =========================
# ONE SEED RUN
# =========================
def run_one_seed(seed, train_ds, test_ds):
    set_seed(seed)

    train_A = class_subset(train_ds, range(50))
    train_B = class_subset(train_ds, range(50, 100))
    test_A = class_subset(test_ds, range(50))
    test_B = class_subset(test_ds, range(50, 100))

    trainA_loader = DataLoader(train_A, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    trainB_loader = DataLoader(train_B, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    testA_loader = DataLoader(test_A, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    testB_loader = DataLoader(test_B, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = CIFARResNet18(num_classes=100).to(DEVICE)
    buffer = ReservoirBuffer(MEMORY_SIZE)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[int(0.6 * (EPOCHS_A + EPOCHS_B)), int(0.85 * (EPOCHS_A + EPOCHS_B))],
        gamma=0.1
    )

    # ---- Phase 1: Task A only
    for _ in range(EPOCHS_A):
        model.train()
        for x, y in trainA_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            buffer.add_batch(x, y)
        scheduler.step()

    acc_A_init_ta = evaluate(model, testA_loader, mode="A")
    acc_A_init_cil = evaluate(model, testA_loader, mode="all")

    # ---- Phase 2: Task B + replay memory
    for _ in range(EPOCHS_B):
        model.train()
        for x_b, y_b in trainB_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)

            if buffer.can_sample(x_b.size(0)):
                x_m, y_m = buffer.sample(x_b.size(0), DEVICE)
                x = torch.cat([x_b, x_m], dim=0)
                y = torch.cat([y_b, y_m], dim=0)
            else:
                x, y = x_b, y_b

            logits = model(x)
            loss = F.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # vanilla ER stores incoming stream samples
            buffer.add_batch(x_b, y_b)
        scheduler.step()

    # Final metrics
    acc_A_final_ta = evaluate(model, testA_loader, mode="A")
    acc_B_final_ta = evaluate(model, testB_loader, mode="B")
    acc_A_final_cil = evaluate(model, testA_loader, mode="all")
    acc_B_final_cil = evaluate(model, testB_loader, mode="all")

    retention_ta = (acc_A_final_ta / acc_A_init_ta) * 100 if acc_A_init_ta > 0 else 0.0
    retention_cil = (acc_A_final_cil / acc_A_init_cil) * 100 if acc_A_init_cil > 0 else 0.0

    forgetting_ta = acc_A_init_ta - acc_A_final_ta
    forgetting_cil = acc_A_init_cil - acc_A_final_cil

    bwt_ta = acc_A_final_ta - acc_A_init_ta
    bwt_cil = acc_A_final_cil - acc_A_init_cil

    return {
        "A_init_ta": acc_A_init_ta,
        "A_final_ta": acc_A_final_ta,
        "B_final_ta": acc_B_final_ta,
        "A_final_cil": acc_A_final_cil,
        "B_final_cil": acc_B_final_cil,
        "retention_ta": retention_ta,
        "retention_cil": retention_cil,
        "forgetting_ta": forgetting_ta,
        "forgetting_cil": forgetting_cil,
        "bwt_ta": bwt_ta,
        "bwt_cil": bwt_cil,
    }

# =========================
# MAIN
# =========================
def main():
    print(f"Device: {DEVICE}")
    print(f"QUICK_RUN={QUICK_RUN} | EPOCHS_A={EPOCHS_A} | EPOCHS_B={EPOCHS_B} | MEMORY_SIZE={MEMORY_SIZE}")
    print(f"Seeds: {SEEDS}")

    root = "/kaggle/input/cifar-100" if os.path.exists("/kaggle/input/cifar-100") else "./data"
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=train_tf)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=test_tf)

    all_results = []
    print("\n[1] Running experiments...")
    for s in SEEDS:
        print(f" -> Seed {s}...")
        r = run_one_seed(s, train_ds, test_ds)
        all_results.append(r)
        print(
            f"    ↳ Seed {s}: "
            f"A_init(TA)={r['A_init_ta']:.2f} | "
            f"A_final(TA)={r['A_final_ta']:.2f} | "
            f"B_final(TA)={r['B_final_ta']:.2f} | "
            f"A_final(CIL)={r['A_final_cil']:.2f} | "
            f"B_final(CIL)={r['B_final_cil']:.2f}"
        )

    def mean_std(key):
        vals = [r[key] for r in all_results]
        return np.mean(vals), np.std(vals)

    print("\n" + "=" * 90)
    print("SUMMARY (Mean ± Std)")
    print("=" * 90)

    m, s = mean_std("A_init_ta")
    print(f"A Init (Task-aware)       : {m:.2f} ± {s:.2f}")
    m, s = mean_std("A_final_ta")
    print(f"A Final (Task-aware)      : {m:.2f} ± {s:.2f}")
    m, s = mean_std("B_final_ta")
    print(f"B Final (Task-aware)      : {m:.2f} ± {s:.2f}")
    m, s = mean_std("A_final_cil")
    print(f"A Final (CIL)             : {m:.2f} ± {s:.2f}")
    m, s = mean_std("B_final_cil")
    print(f"B Final (CIL)             : {m:.2f} ± {s:.2f}")

    m, s = mean_std("retention_ta")
    print(f"Retention (Task-aware)    : {m:.2f} ± {s:.2f}")
    m, s = mean_std("retention_cil")
    print(f"Retention (CIL)           : {m:.2f} ± {s:.2f}")
    m, s = mean_std("forgetting_ta")
    print(f"Forgetting (Task-aware)   : {m:.2f} ± {s:.2f}")
    m, s = mean_std("forgetting_cil")
    print(f"Forgetting (CIL)          : {m:.2f} ± {s:.2f}")
    m, s = mean_std("bwt_ta")
    print(f"BWT (Task-aware)          : {m:.2f} ± {s:.2f}")
    m, s = mean_std("bwt_cil")
    print(f"BWT (CIL)                 : {m:.2f} ± {s:.2f}")

    print("=" * 90)

if __name__ == "__main__":
    main()

Device: cuda
QUICK_RUN=False | EPOCHS_A=50 | EPOCHS_B=100 | MEMORY_SIZE=2000
Seeds: [0, 1, 2, 3, 4]


100%|██████████| 169M/169M [00:02<00:00, 78.1MB/s]



[1] Running experiments...
 -> Seed 0...
    ↳ Seed 0: A_init(TA)=50.50 | A_final(TA)=38.06 | B_final(TA)=64.06 | A_final(CIL)=5.78 | B_final(CIL)=64.04
 -> Seed 1...
    ↳ Seed 1: A_init(TA)=52.20 | A_final(TA)=38.40 | B_final(TA)=64.16 | A_final(CIL)=6.58 | B_final(CIL)=64.14
 -> Seed 2...
    ↳ Seed 2: A_init(TA)=50.16 | A_final(TA)=41.64 | B_final(TA)=67.34 | A_final(CIL)=6.02 | B_final(CIL)=67.24
 -> Seed 3...
    ↳ Seed 3: A_init(TA)=50.20 | A_final(TA)=41.58 | B_final(TA)=66.10 | A_final(CIL)=6.18 | B_final(CIL)=66.10
 -> Seed 4...
    ↳ Seed 4: A_init(TA)=51.60 | A_final(TA)=42.22 | B_final(TA)=68.28 | A_final(CIL)=6.80 | B_final(CIL)=68.26

SUMMARY (Mean ± Std)
A Init (Task-aware)       : 50.93 ± 0.82
A Final (Task-aware)      : 40.38 ± 1.77
B Final (Task-aware)      : 65.99 ± 1.68
A Final (CIL)             : 6.27 ± 0.37
B Final (CIL)             : 65.96 ± 1.67
Retention (Task-aware)    : 79.32 ± 4.02
Retention (CIL)           : 12.31 ± 0.58
Forgetting (Task-aware)   : 10.55 